# Initial exact Chicago restaurant merge

This notebook previews the outputs of `build_initial_merge.py`. Run that script first. The scope is all Chicago restaurants; only exact, corroborated, unambiguous links enter the primary merged table. See `README.md` for the acceptance and date policies.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "merge_plan.md").exists():
    ROOT = ROOT / "ChicagoPizza514"
DATA = ROOT / "data" / "processed" / "initial_exact"
audit = json.loads((ROOT / "data" / "reports" / "initial_merge_audit.json").read_text())
assert audit["status"] == "complete", "Run build_initial_merge.py to completion first"
assert all(audit["acceptance_checks"].values())
audit["parameters"]


## Primary merged dataset

One row per accepted restaurant episode, Google source, and Chicago calendar month. The inspection strictly precedes the month; review aggregates use only eligible deduplicated records. Months with tied, unscored, stale, or insufficiently bracketed inspection evidence are excluded. The full monthly table is small enough to load; source previews below are limited.


In [ ]:
restaurant_month = pd.read_parquet(DATA / "restaurant_month.parquet")
print(f"{len(restaurant_month):,} restaurant-month rows")
print(f"{restaurant_month.restaurant_id.nunique():,} restaurants")
restaurant_month.head(20)


In [ ]:
pd.Series(audit["coverage"], name="count").to_frame()


In [ ]:
pd.Series(audit["monthly"]["decisions"], name="candidate_months").to_frame()


In [ ]:
pd.Series(audit["reviews"]["eligibility_first_exclusion"], name="deduplicated_reviews").to_frame()


## Inspect the matching evidence

Accepted means the documented source checks passed. It does not prove ownership or opening/closing dates. Observed first/last inspections are not assumed to be business validity dates. Full source inspection IDs and raw name/address evidence are retained in the crosswalk.


In [ ]:
accepted_links = pd.read_csv(ROOT / "data" / "reports" / "initial_merge_accepted_links.csv", dtype="string")
accepted_links[["google_name", "google_address", "license", "distance_max_m", "inspection_events", "decision"]].head(20)


## Small previews of the scoped source tables

These tables preserve unmatched businesses and records excluded from the primary analysis. `eligible_for_monthly_candidate` is checked before the month-level rules; use `monthly_eligibility_audit.csv` to determine final month inclusion. Row counts below are full Parquet counts; displayed rows are previews only.


In [ ]:
from IPython.display import display

for filename in ("chicago_restaurants.parquet", "chicago_restaurant_reviews.parquet", "chicago_restaurant_inspections.parquet"):
    file = pq.ParquetFile(DATA / filename)
    print(filename, f"{file.metadata.num_rows:,} total rows")
    first_batch = next(file.iter_batches(batch_size=5), None)
    if first_batch is not None:
        display(first_batch.to_pandas())


## Interpretation limits

This is a deliberately small retrospective cohort. Whole-history identity checks and observation bracketing can select stable businesses using later evidence. Revisit cohort selection before prospective prediction, even though every joined inspection predates its month. Undated metadata averages/status are not monthly predictors. Missing review months are not proof of zero activity, and conditional passes remain distinct from passes and failures.


In [ ]:
pd.Series(audit["acceptance_checks"], name="passed").to_frame()
